# 03 · Station choice, network load and queues

**Question.** If drivers choose among today's rapid sites by travel time, price, power and size, where does the demand land, and where do queues form at the peak hour?

**Method.** Multinomial logit allocation (`chargescape.models.choice`, literature-informed parameters) of rapid-type demand (destination + PHV, P50) from every hex to rapid sites within 20 minutes on the OSM drive network; Erlang-C M/M/c queue per site with 38-minute service (Dundee median). Supply is calibrated so that each borough's rapid ports match DfT's official 50 kW+ counts (OSM alone sees only ~15 % of them).

In [ ]:
import warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from chargescape.utils.paths import PROCESSED, INTERIM, FIGURES, TABLES, DOCS
from chargescape.utils.io import read_gdf
from chargescape.viz.figures import hex_map, save, barh, boroughs
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
feats = read_gdf(PROCESSED / "hex_features_r8.small.parquet")
print(feats.shape)

In [ ]:
from chargescape.models.choice import ChoiceParams
print(ChoiceParams().__dict__)
sites = pd.read_parquet(PROCESSED / "rapid_sites_load_2026.small.parquet")
svc = pd.read_parquet(PROCESSED / "hex_service_2026.small.parquet")
print(f"{len(sites)} rapid sites, {sites.ports.sum():,} ports after calibration (known ports {sites.ports_known.sum():,.0f}, median scale ×{sites.port_scale.median():.1f})")
sites[["sessions_day", "kwh_day", "utilisation_24h", "utilisation_peak", "p_wait", "mean_wait_min"]].describe().round(2).T

## 1. Where load concentrates and where queues form

In [ ]:
s = gpd.GeoDataFrame(sites, geometry=gpd.points_from_xy(sites.lon, sites.lat), crs=4326).to_crs(27700)
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
boroughs().boundary.plot(ax=axes[0], color="#bbbbbb", linewidth=0.4)
s.plot(ax=axes[0], markersize=np.sqrt(s.sessions_day) * 2, column="utilisation_24h", cmap="viridis", vmin=0, vmax=0.6, alpha=0.85, legend=True, legend_kwds={"label": "24h utilisation", "shrink": 0.6})
axes[0].set_axis_off(); axes[0].set_title("Rapid sites: size = expected sessions/day, colour = utilisation", loc="left")
boroughs().boundary.plot(ax=axes[1], color="#bbbbbb", linewidth=0.4)
s.plot(ax=axes[1], markersize=14, column="p_wait", cmap="Reds", vmin=0, vmax=1, legend=True, legend_kwds={"label": "P(wait) at peak hour", "shrink": 0.6})
axes[1].set_axis_off(); axes[1].set_title("Probability of waiting at the peak hour (Erlang C)", loc="left")
plt.tight_layout(); save(fig, "03_site_load_queues.png"); plt.show()
print(f"sites with P(wait) > 0.5 at peak: {(sites.p_wait > 0.5).sum()} of {len(sites)}; median expected wait {sites.mean_wait_min.replace(np.inf, np.nan).median():.1f} min")
sites.sort_values("sessions_day", ascending=False)[["borough", "operator", "power_kw", "ports", "sessions_day", "utilisation_24h", "p_wait", "mean_wait_min"]].head(10).round(2)

## 2. What drivers experience, by hex

In [ ]:
g = feats.merge(svc, on="h3", how="left")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
hex_map(g, "expected_wait_min", "Expected wait faced by a hex's drivers at the peak hour (min)", cmap="YlOrRd", vmin=0, vmax=30, ax=axes[0], legend_label="min")
hex_map(g, "n_rapid_within_10min", "Rapid sites reachable within 10 min drive", cmap="Blues", scheme="quantiles", k=6, ax=axes[1], legend_label="sites")
plt.tight_layout(); save(fig, "03_hex_experience.png"); plt.show()

## 3. Incremental demand vs cannibalisation for a candidate hub

In [ ]:
from chargescape.models.choice import utilities, incremental_demand, choice_probabilities
from chargescape.features.travel_time import travel_time_minutes
from chargescape.models.allocation_pipeline import rapid_sites, KWH_PER_RAPID_SESSION
rs = rapid_sites()
tt, method = travel_time_minutes(feats, rs, name="hex_r8_to_rapid")
tt = np.where(np.isfinite(tt), tt, 1e6)
d26 = pd.read_parquet(PROCESSED / "demand_r8.small.parquet"); d26 = d26[d26.scenario == "2026"].set_index("h3").reindex(feats.h3)
demand = (d26.dest_kwh_p50.fillna(0) + d26.phv_kwh_p50.fillna(0)).to_numpy() / KWH_PER_RAPID_SESSION / 365
p = ChoiceParams(outside_utility=0.0)
v = utilities(tt, rs.price.to_numpy(), rs.power_kw.to_numpy(), rs.ports.to_numpy(), None, p)
# candidate: the hex with the largest unserved demand
svc_i = svc.set_index("h3").reindex(feats.h3)
cand_idx = int(np.argmax(svc_i.unserved_sessions_day.fillna(0).to_numpy() + demand * np.clip(svc_i.expected_wait_min.fillna(0).to_numpy() / 30, 0, 1)))
cand = feats.iloc[[cand_idx]][["h3", "lat", "lon", "borough"]]
ttc, _ = travel_time_minutes(feats, cand, name="hex_r8_to_democand", cache=False)
ttc = np.where(np.isfinite(ttc), ttc, 1e6)
rows = []
for kw, ports in [(50, 2), (150, 4), (150, 8), (300, 8)]:
    vc = utilities(ttc, np.array([79.0]), np.array([float(kw)]), np.array([float(ports)]), None, p)[:, 0]
    r = incremental_demand(demand, v, vc, outside_utility=0.0); r.update({"power_kw": kw, "ports": ports}); rows.append(r)
print("candidate hub in", cand.borough.iloc[0], cand.h3.iloc[0])
pd.DataFrame(rows)[["power_kw", "ports", "captured", "new", "cannibalised", "cannibalisation_share"]].round(1)

**Reading.** Even at the hex with the largest unmet demand, a new hub takes a substantial share of its sessions from existing sites; bigger and faster hubs capture more in total but also cannibalise more. This is the number an investor or partner needs next to the headline "expected sessions", and it is exactly what a pure gap map cannot show. Parameters are literature-informed; the sensitivity of siting to them is reported in notebook 04.